# Data Ingestion
This notebook demonstrates loading a CSV file into the project raw-data folder.
Explore the data to understand the following
- Check all the categorical column to know the count of unique values using
   df["col"].value_counts()
   may need to normalize some value like man/MAN/Man to be small letter man
   df["col"] = df["col"].str.lower()
- For int check if values have data to be considered out of range / outliers
   max_val = max(df["col"])
   min_val = min(df["col"])
- Using darts
If our problem has just the period and target
    from darts import TimeSeries
    from darts.models import RegressionModel
    from sklearn.linear_model import LinearRegression

    y = TimeSeries(df["y"])
    y_train = y[:-24]

    model = RegressionModel(
        lags=[-1, -7, -28]
        model=LinearRegression()
    )

    model.fit(series=y_train)
    y_predict = model.predict(n=24, series=y_train)

if we have one or more exogenous features
    y = TimeSeries.from_series(df["y"])
    features = ["col_1", "col_2", ...]
    future_cov = TimeSeries.from_dataframe(df[features])

    model = RegressionModel(
        lags = [-1, -7, 28],
        lags_future_covariates = [0],
        model = LinearRegression()
    )

    model.fit(y, future_covariates=future_cov)

    y_pred = model.predict(
        n=24,
        series=y,
        future_covariates=future_cov
    )

If we have multi-timeseries forecasting:
    y_all = TimeSeries.from_group_dataframe(
        df,
        group_cols =["col_1", "col_2", ..],
        time_col = "datetime_col",
        value_col = ["y"]
    )

    future_cov = TimeSeries.from_group_dataframe(
        df,
        group_cols =["col_1", "col_2", ..],
        time_col = "datetime_col",
        value_col = ["col_1", "col_2", ...]
    )

    model=RegressionModel(
        lags = [1,7,28],
        lags_future_covariates=[0,1,2],
        model=LinearRegression()
    )

    model.fit(y_all, future_covariates=future_cov)

    model.predict(n=30, series=y_all, future_covariates=future_cov)


In [14]:
import importlib
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "src" / "mlops_end_2_end").is_dir() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import mlops_end_2_end.data.data_ingestion.data_ingestion as data_ingestion_module
importlib.reload(data_ingestion_module)
DataIngestion = data_ingestion_module.DataIngestion

_path = project_root / "notebooks" / "sales_timeseries_2025.csv"
ingestor = DataIngestion(project_root=project_root)

In [15]:
# Example usage when a real source file exists:
df = ingestor.load_data(_path)
df.head()

2026-10-04 21:56:14 - data_ingestion - INFO - Starting CSV ingestion for C:\Users\HP-USER\MLOps-end-2-end\notebooks\sales_timeseries_2025.csv
2026-10-04 21:56:14 - data_ingestion - INFO - The provided path is a local file: C:\Users\HP-USER\MLOps-end-2-end\notebooks\sales_timeseries_2025.csv
2026-10-04 21:56:14 - data_ingestion - INFO - Reading CSV into pandas DataFrame
2026-10-04 21:56:14 - data_ingestion - INFO - Saving ingested data to C:\Users\HP-USER\MLOps-end-2-end\data\raw\sales_timeseries_2025.csv
2026-10-04 21:56:14 - data_ingestion - INFO - CSV ingestion completed successfully for C:\Users\HP-USER\MLOps-end-2-end\notebooks\sales_timeseries_2025.csv


,date,sales,price,promotion,temperature,holiday,day_of_week,month,week_of_year,trend
0,2025-01-01,89.30,10.08,0,3.10,1,2,1,1,100.000000
1,2025-01-02,117.17,8.70,1,2.04,0,3,1,1,100.137363
2,2025-01-03,95.25,10.07,0,1.33,0,4,1,1,100.274725
3,2025-01-04,114.66,9.61,0,2.76,0,5,1,1,100.412088
4,2025-01-05,112.80,10.34,0,4.16,0,6,1,1,100.549451


In [13]:
# check data schema and other properties
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          365 non-null    str    
 1   sales         365 non-null    float64
 2   price         365 non-null    float64
 3   promotion     365 non-null    int64  
 4   temperature   365 non-null    float64
 5   holiday       365 non-null    int64  
 6   day_of_week   365 non-null    int64  
 7   month         365 non-null    int64  
 8   week_of_year  365 non-null    int64  
 9   trend         365 non-null    float64
dtypes: float64(4), int64(5), str(1)
memory usage: 32.2 KB
